In [13]:
# General
import numpy as np
import pandas as pd

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Model Selection
from sklearn.model_selection import train_test_split

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)

# Imbalance Handling
from imblearn.over_sampling import SMOTE

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
# Loading Dataset
creditcard_dataset = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")

In [15]:
# Standarizing Data
scaler = StandardScaler()
creditcard_dataset["Amount"] = scaler.fit_transform(creditcard_dataset[["Amount"]])

In [16]:
# Standarizing Time
creditcard_dataset["Hour"] = (creditcard_dataset["Time"] // 3600) % 24
creditcard_dataset.drop("Time", axis=1, inplace=True)

In [17]:
# Setting Target Coloumn
X = creditcard_dataset.drop("Class", axis=1)
Y = creditcard_dataset["Class"]

In [18]:
# Train-Text Split
X_train, X_test, Y_train, Y_test = train_test_split( X, Y, test_size=0.2,stratify=Y, random_state=2)

In [19]:
# Calculating Ratio
ratio = len(Y_train[Y_train == 0]) / len(Y_train[Y_train == 1])

In [20]:
# Train XGB
model = XGBClassifier(
    scale_pos_weight=ratio,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='auc'
)

model.fit(X_train, Y_train)

# Predictions
Y_pred = model.predict(X_test)
Y_prob = model.predict_proba(X_test)[:, 1]

In [21]:
def evaluate_model(name, y_test, y_pred, y_prob):
    print("="*50)
    print(f"{name} RESULTS")
    print("="*50)
    
    print("\n📊 Classification Report:\n")
    print(classification_report(Y_test, Y_pred, digits=4))
    
    roc = roc_auc_score(Y_test, Y_prob)
    print("📈 ROC-AUC Score:", round(roc, 4))
    
    cm = confusion_matrix(Y_test, Y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print("\n🔍 Confusion Matrix:")
    print(cm)
    
    print("\nBreakdown:")
    print(f"TP (Fraud detected): {tp}")
    print(f"FN (Fraud missed): {fn}")
    print(f"FP (False alarm): {fp}")
    print(f"TN (Correct normal): {tn}")
    
    print("="*50)

In [22]:
# XGBoost
evaluate_model("XGBoost", Y_test, Y_pred, Y_prob)

XGBoost RESULTS

📊 Classification Report:

              precision    recall  f1-score   support

           0     0.9998    0.9998    0.9998     56864
           1     0.8842    0.8571    0.8705        98

    accuracy                         0.9996     56962
   macro avg     0.9420    0.9285    0.9351     56962
weighted avg     0.9996    0.9996    0.9996     56962

📈 ROC-AUC Score: 0.9899

🔍 Confusion Matrix:
[[56853    11]
 [   14    84]]

Breakdown:
TP (Fraud detected): 84
FN (Fraud missed): 14
FP (False alarm): 11
TN (Correct normal): 56853
